# 🧠 Advanced Data Augmentation with Keras

> **Module:** 03 — Deep Learning with Keras and TensorFlow  
> **Topic:** Data augmentation, normalisation, custom preprocessing  
> **Dataset:** CIFAR-10 (60,000 colour images, 10 classes)

---

## 📋 Overview

In this notebook I implement and experiment with **data augmentation** — a family of techniques that artificially expand a training dataset by applying controlled transformations to existing images.

| Part | Technique | Tool |
|---|---|---|
| Part 1 | Basic geometric augmentation | `ImageDataGenerator` |
| Part 2 | Feature-wise & sample-wise normalisation | `ImageDataGenerator.fit()` |
| Part 3 | Custom augmentation function (random noise) | `preprocessing_function=` |
| Part 4 | Visualise augmented images | `matplotlib` |
| Exercises | Apply all three techniques to CIFAR-10 images | Solved below |

## 🧩 Theory

### Why augment?

A model trained on a small fixed dataset memorises the exact pixel patterns it saw — it overfits. Data augmentation generates **new training samples on-the-fly**, so the model never sees the exact same image twice.

**Telecom / RF analogy 📡:** Data augmentation = synthetically generating diverse channel conditions during training. Rotation ≈ phase rotation, zoom ≈ delay spread variation, noise injection ≈ AWGN.

### Normalisation

**Feature-wise:** $x'_{i} = (x_i - \mu_{\text{dataset}}) / \sigma_{\text{dataset}}$

**Sample-wise:** $x'_{i} = (x_i - \mu_{\text{sample}}) / \sigma_{\text{sample}}$

**Gaussian noise:** $x'_i = x_i + \epsilon_i,\ \epsilon_i \sim \mathcal{N}(0, \sigma^2)$

## ⚙️ Part 0 — Setup & Dataset

In [ ]:
# Install required libraries
# 'datasets' = HuggingFace library: fast CDN download, no protobuf/tensorflow-metadata dependency
!pip install tensorflow==2.16.2 matplotlib==3.9.1 scipy datasets --quiet

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from datasets import load_dataset

print(f"TensorFlow version: {tf.__version__}")

# Load CIFAR-10 via HuggingFace datasets (fast CDN, no protobuf dependency)
ds = load_dataset('uoft-cs/cifar10', trust_remote_code=True)

x_train = np.array([img for img in ds['train']['img']])
y_train = np.array(ds['train']['label']).reshape(-1, 1)
x_test  = np.array([img for img in ds['test']['img']])
y_test  = np.array(ds['test']['label']).reshape(-1, 1)

x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print(f"Training set:  {x_train.shape}  ({x_train.dtype})")
print(f"Test set:      {x_test.shape}   ({x_test.dtype})")
print(f"Pixel range:   [{x_train.min():.2f}, {x_train.max():.2f}]")

In [ ]:
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

plt.figure(figsize=(10, 10))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[y_train[i][0]], fontsize=8)
    plt.axis('off')
plt.suptitle('📥 CIFAR-10 — Raw Training Images', fontsize=14)
plt.tight_layout(); plt.show()

### 🖼️ Synthetic Sample Image

In [ ]:
from PIL import Image, ImageDraw

image = Image.new('RGB', (224, 224), color=(255, 255, 255))
draw  = ImageDraw.Draw(image)
draw.rectangle([(50, 50), (174, 174)], fill=(255, 0, 0))
image.save('sample.jpg')

plt.imshow(image); plt.title('Synthetic sample image'); plt.axis('off'); plt.show()
print('sample.jpg saved.')

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

img = load_img('sample.jpg')
x   = np.expand_dims(img_to_array(img), axis=0)  # (1, 224, 224, 3)
print(f"Image tensor shape: {x.shape}")

## 🔄 Part 1 — Basic Geometric Augmentation

| Parameter | Value | Effect |
|---|---|---|
| `rotation_range` | 40° | Random rotation in $[-40°, +40°]$ |
| `width_shift_range` | 0.2 | Shift left/right up to 20% |
| `height_shift_range` | 0.2 | Shift up/down up to 20% |
| `shear_range` | 0.2 | Shear up to 0.2 rad |
| `zoom_range` | 0.2 | Zoom factor in [0.8, 1.2] |
| `horizontal_flip` | True | Random horizontal mirror |
| `fill_mode` | 'nearest' | Fill new pixels via nearest neighbour |

In [ ]:
datagen_basic = ImageDataGenerator(
    rotation_range=40, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_basic.flow(x, batch_size=1)):
    axes[i].imshow(batch[0].astype('uint8')); axes[i].set_title(f'🔄 Aug {i+1}'); axes[i].axis('off')
    if i >= 3: break
plt.suptitle('Part 1 — Basic Geometric Augmentations', fontsize=12); plt.tight_layout(); plt.show()

## 📐 Part 2 — Feature-wise & Sample-wise Normalisation

**Feature-wise** computes $\mu$/$\sigma$ across the full dataset (requires `datagen.fit()`).  
**Sample-wise** computes $\mu$/$\sigma$ per individual image (no fit needed).

In [ ]:
datagen_norm = ImageDataGenerator(
    featurewise_center=True, featurewise_std_normalization=True,
    samplewise_center=True, samplewise_std_normalization=True
)
datagen_norm.fit(x)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_norm.flow(x, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    axes[i].set_title(f'📐 Norm {i+1}'); axes[i].axis('off')
    if i >= 3: break
plt.suptitle('Part 2 — Feature-wise & Sample-wise Normalisation', fontsize=12); plt.tight_layout(); plt.show()

## ⚡ Part 3 — Custom Augmentation: Gaussian Noise

Custom `preprocessing_function` — called on every batch. Injects AWGN: $x'_i = x_i + \epsilon_i,\ \epsilon_i \sim \mathcal{N}(0, 0.01)$.

In [ ]:
def add_random_noise(image):
    """Add AWGN: x' = x + N(0, 0.1²)"""
    return image + np.random.normal(0.0, 0.1, image.shape)

datagen_noise = ImageDataGenerator(preprocessing_function=add_random_noise)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_noise.flow(x, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    axes[i].set_title(f'⚡ Noisy {i+1}'); axes[i].axis('off')
    if i >= 3: break
plt.suptitle('Part 3 — Gaussian Noise (σ=0.1)', fontsize=12); plt.tight_layout(); plt.show()

## 📊 Part 4 — Visualise Augmented Variants

In [ ]:
plt.figure(figsize=(8, 8))
for i, batch in enumerate(datagen_noise.flow(x, batch_size=1)):
    if i >= 4: break
    plt.subplot(2, 2, i + 1)
    plt.imshow(np.clip(batch[0], 0, 255).astype('uint8'))
    plt.title(f'🧪 Version {i+1}'); plt.axis('off')
plt.suptitle('📊 Augmented Variants (Gaussian Noise)', fontsize=13); plt.tight_layout(); plt.show()

---
## 🔢 Exercises — Solved

All three exercises use the CIFAR-10 images already loaded above — no additional download required.

In [ ]:
from tensorflow.keras.preprocessing.image import array_to_img

# Use first 3 CIFAR-10 training images — already loaded, no external download needed
training_images = (x_train[:3] * 255.0).astype('float32')  # (3, 32, 32, 3)
print(f"Training batch shape: {training_images.shape}")

datagen_ex1 = ImageDataGenerator(
    rotation_range=40, width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex1.flow(training_images, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8')); axes[i].set_title(f'🔄 Aug {i+1}'); axes[i].axis('off')
    if i >= 3: break
plt.suptitle('✅ Exercise 1 — Geometric Augmentation (CIFAR-10)', fontsize=13); plt.tight_layout(); plt.show()

In [ ]:
datagen_ex2 = ImageDataGenerator(featurewise_center=True, featurewise_std_normalization=True,
                                   samplewise_center=True, samplewise_std_normalization=True)
datagen_ex2.fit(training_images)
print(f"Mean: {datagen_ex2.mean}  Std: {datagen_ex2.std}")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex2.flow(training_images, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8')); axes[i].set_title(f'📐 Norm {i+1}'); axes[i].axis('off')
    if i >= 3: break
plt.suptitle('✅ Exercise 2 — Normalisation (CIFAR-10)', fontsize=13); plt.tight_layout(); plt.show()

In [ ]:
def add_random_noise(image):
    return image + np.random.normal(0.0, 0.1, image.shape)

datagen_ex3 = ImageDataGenerator(preprocessing_function=add_random_noise)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, batch in enumerate(datagen_ex3.flow(training_images, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8')); axes[i].set_title(f'⚡ Noisy {i+1}'); axes[i].axis('off')
    if i >= 3: break
plt.suptitle('✅ Exercise 3 — Gaussian Noise (CIFAR-10)', fontsize=13); plt.tight_layout(); plt.show()

---
## 📊 Summary

| Technique | Key call | Formula |
|---|---|---|
| Geometric augmentation | `ImageDataGenerator(rotation_range=..., ...)` | $T: (x,y) \rightarrow (x', y')$ |
| Feature-wise normalisation | `featurewise_center=True` + `.fit()` | $x' = (x - \mu) / \sigma$ |
| Sample-wise normalisation | `samplewise_center=True` | $x' = (x - \mu_{\text{img}}) / \sigma_{\text{img}}$ |
| Custom noise | `preprocessing_function=fn` | $x' = x + \mathcal{N}(0, \sigma^2)$ |

**Augmentation as regularisation:** $\mathcal{L}_{\text{eff}} = \mathbb{E}_{T \sim p(T)}[\mathcal{L}(f(T(x)), y)]$

---
## 🧪 Sandbox

In [ ]:
def add_gaussian_noise(image, sigma=0.05):
    return np.clip(image + np.random.normal(0, sigma, image.shape), 0, 255)

datagen_combined = ImageDataGenerator(rotation_range=30, zoom_range=0.15, horizontal_flip=True,
                                       fill_mode='reflect', preprocessing_function=add_gaussian_noise)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, batch in enumerate(datagen_combined.flow(x, batch_size=1)):
    r, c = divmod(i, 4)
    axes[r][c].imshow(np.clip(batch[0], 0, 255).astype('uint8')); axes[r][c].set_title(f'Combined {i+1}'); axes[r][c].axis('off')
    if i >= 7: break
plt.suptitle('🧪 Sandbox — Geometric + Noise', fontsize=13); plt.tight_layout(); plt.show()

In [ ]:
def iq_noise_augmentation(image):
    img = image.copy()
    img[:,:,0] += np.random.normal(0, 15.0, image[:,:,0].shape)  # I
    img[:,:,1] += np.random.normal(0, 15.0, image[:,:,1].shape)  # Q
    img[:,:,2] += np.random.normal(0,  5.0, image[:,:,2].shape)  # carrier
    return img

datagen_iq = ImageDataGenerator(preprocessing_function=iq_noise_augmentation)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, batch in enumerate(datagen_iq.flow(x, batch_size=1)):
    axes[i].imshow(np.clip(batch[0], 0, 255).astype('uint8')); axes[i].set_title(f'📡 IQ {i+1}'); axes[i].axis('off')
    if i >= 3: break
plt.suptitle('🧪 IQ Channel Noise (R=I, G=Q, B=carrier)', fontsize=13); plt.tight_layout(); plt.show()

In [ ]:
cifar_batch = x_train[:4] * 255.0
datagen_cifar = ImageDataGenerator(rotation_range=20, horizontal_flip=True, zoom_range=0.1, fill_mode='nearest')
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j in range(4):
    axes[0][j].imshow(x_train[j]); axes[0][j].set_title(CLASS_NAMES[y_train[j][0]]); axes[0][j].axis('off')
for i, batch in enumerate(datagen_cifar.flow(cifar_batch, batch_size=4)):
    for j in range(4):
        axes[1][j].imshow(np.clip(batch[j], 0, 255).astype('uint8')); axes[1][j].set_title('Aug'); axes[1][j].axis('off')
    break
plt.suptitle('🧪 Original vs Augmented CIFAR-10', fontsize=13); plt.tight_layout(); plt.show()